In [ ]:
!pip -q install -U \
  "transformers==4.41.2" \
  "tokenizers==0.19.1" \
  "datasets==2.20.0" \
  "accelerate==0.33.0" \
  "peft==0.12.0" \
  sentencepiece

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA device count: 1
GPU: Tesla T4


In [3]:
import transformers
import tokenizers
import datasets
import accelerate
import peft

print("transformers:", transformers.__version__)
print("tokenizers  :", tokenizers.__version__)
print("datasets    :", datasets.__version__)
print("accelerate  :", accelerate.__version__)
print("peft        :", peft.__version__)

transformers: 4.41.2
tokenizers  : 0.19.1
datasets    : 2.20.0
accelerate  : 0.33.0
peft        : 0.12.0


In [4]:
!ls /kaggle/input/datasets/mahithgangu/security-patch-dataset-v1/security_patch_dataset_v1

train.jsonl  val.jsonl


In [5]:
MODEL_NAME = "Salesforce/codet5-base"

TRAIN_FILE = "/kaggle/input/datasets/mahithgangu/security-patch-dataset-v1/security_patch_dataset_v1/train.jsonl"
VAL_FILE   = "/kaggle/input/datasets/mahithgangu/security-patch-dataset-v1/security_patch_dataset_v1/val.jsonl"

WORK = Path("/kaggle/working")
OUT_DIR = WORK / "models" / "codet5_security_adapter" / "adapter"
RUN_DIR = WORK / "security_adapter_runs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_FILE:", TRAIN_FILE)
print("VAL_FILE  :", VAL_FILE)
print("OUT_DIR   :", OUT_DIR)
print("RUN_DIR   :", RUN_DIR)

TRAIN_FILE: /kaggle/input/datasets/mahithgangu/security-patch-dataset-v1/security_patch_dataset_v1/train.jsonl
VAL_FILE  : /kaggle/input/datasets/mahithgangu/security-patch-dataset-v1/security_patch_dataset_v1/val.jsonl
OUT_DIR   : /kaggle/working/models/codet5_security_adapter/adapter
RUN_DIR   : /kaggle/working/security_adapter_runs


In [6]:
from datasets import load_dataset

raw = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE,
    }
)

print(raw)
print(raw["train"][0])

DatasetDict({
    train: Dataset({
        features: ['task', 'split_hint', 'family', 'category', 'error_line', 'input', 'target', 'metadata'],
        num_rows: 24000
    })
    validation: Dataset({
        features: ['task', 'split_hint', 'family', 'category', 'error_line', 'input', 'target', 'metadata'],
        num_rows: 2400
    })
})
{'task': 'security_repair_line', 'split_hint': 'train', 'family': 'scanf_percent_s_bounded', 'category': 'UNBOUNDED_INPUT', 'error_line': 5, 'input': 'SECURITY_FIX\nCATEGORY UNBOUNDED_INPUT\nLINE 5\n\n<CODE>\n#include <stdio.h>\n\nvoid read_data(void) {\n    char name[8];\n    scanf("%s", name);\n    printf("%s\\n", name);\n}\n</CODE>', 'target': 'REPLACE_LINE 5 scanf("%7s", name);', 'metadata': {'vulnerable_line': 'scanf("%s", name);', 'fixed_line': 'scanf("%7s", name);'}}


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.")

Tokenizer loaded.


In [8]:
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 64

def preprocess(batch):
    model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tok = raw.map(
    preprocess,
    batched=True,
    remove_columns=raw["train"].column_names
)

print(tok)
print("Example input len :", len(tok["train"][0]["input_ids"]))
print("Example target len:", len(tok["train"][0]["labels"]))

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 24000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2400
    })
})
Example input len : 72
Example target len: 14


In [9]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q", "v"],
)

model = get_peft_model(base_model, lora)
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 224,651,520 || trainable%: 0.7877


In [15]:
def normalize_text(s: str) -> str:
    s = (s or "").strip().upper()
    s = re.sub(r"\s+", " ", s)
    return s

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.where(preds >= tokenizer.vocab_size, tokenizer.pad_token_id, preds)
    preds = preds.astype(np.int64)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    labels = np.where(labels >= tokenizer.vocab_size, tokenizer.pad_token_id, labels)
    labels = labels.astype(np.int64)

    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    gold_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_text = [normalize_text(x) for x in pred_text]
    gold_text = [normalize_text(x) for x in gold_text]

    exact = float(np.mean([p == g for p, g in zip(pred_text, gold_text)]))
    return {"exact_match": exact}

In [16]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 3e-4
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
EPOCHS = 2

args = Seq2SeqTrainingArguments(
    output_dir=str(WORK / "training_output"),
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    save_total_limit=2,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=1,

    fp16=False,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="exact_match",
    greater_is_better=True,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")
print("Train rows:", len(tok["train"]))
print("Val rows  :", len(tok["validation"]))

Trainer ready.
Train rows: 24000
Val rows  : 2400


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [17]:
with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "train_file": TRAIN_FILE,
        "val_file": VAL_FILE,
        "max_source_len": MAX_SOURCE_LEN,
        "max_target_len": MAX_TARGET_LEN,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "epochs": EPOCHS,
        "seed": SEED,
    }, f, indent=2)

print("Saved:", RUN_DIR / "config.json")

Saved: /kaggle/working/security_adapter_runs/config.json


In [18]:
trainer.train()

Step,Training Loss,Validation Loss,Exact Match
500,0.000700,0.000020,1.000000
1000,0.000100,0.000004,1.000000
1500,0.000100,0.000003,1.000000
2000,0.000100,0.000002,1.000000
2500,0.000000,0.000002,1.000000
3000,0.000000,0.000002,1.000000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in ver

TrainOutput(global_step=3000, training_loss=0.00044804654456675054, metrics={'train_runtime': 3890.5377, 'train_samples_per_second': 12.338, 'train_steps_per_second': 0.771, 'total_flos': 5381551292350464.0, 'train_loss': 0.00044804654456675054, 'epoch': 2.0})

In [19]:
metrics = trainer.evaluate()
print(metrics)

with open(RUN_DIR / "eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", RUN_DIR / "eval_metrics.json")

{'eval_loss': 2.040517756540794e-05, 'eval_exact_match': 1.0, 'eval_runtime': 357.2054, 'eval_samples_per_second': 6.719, 'eval_steps_per_second': 1.68, 'epoch': 2.0}
Saved: /kaggle/working/security_adapter_runs/eval_metrics.json


In [20]:
trainer.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

print("Saved adapter to:", OUT_DIR)
print("Files:", os.listdir(OUT_DIR))

Saved adapter to: /kaggle/working/models/codet5_security_adapter/adapter
Files: ['README.md', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'adapter_config.json', 'merges.txt', 'tokenizer.json', 'adapter_model.safetensors']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
!zip -r /kaggle/working/security_adapter_runs.zip /kaggle/working/security_adapter_runs > /dev/null
!zip -r /kaggle/working/codet5_security_adapter.zip /kaggle/working/models/codet5_security_adapter > /dev/null
!ls -lah /kaggle/working/*.zip

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


-rw-r--r-- 1 root root 7.0M Mar 23 08:26 /kaggle/working/codet5_security_adapter.zip
-rw-r--r-- 1 root root 1.1K Mar 23 08:26 /kaggle/working/security_adapter_runs.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
